#### Part 1: Exploration and Data Quality
Olist marketplace dataset. Goal: an order-level table I'd be willing to model on.

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")

FILES = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geo": "olist_geolocation_dataset.csv",
    "cat_en": "product_category_name_translation.csv",
}

d = {k: pd.read_csv(RAW / v) for k, v in FILES.items()}

In [2]:
pd.DataFrame(
    {k: {"rows": v.shape[0], "cols": v.shape[1]} for k, v in d.items()}
).T

,rows,cols
orders,99441,8
items,112650,7
payments,103886,5
reviews,99224,7
customers,99441,5
sellers,3095,4
products,32951,9
geo,1000163,5
cat_en,71,2


In [3]:
orders = d["orders"].copy()
date_cols = [c for c in orders.columns if c.endswith("_timestamp") or c.endswith("_date") or c.endswith("_at")]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

orders[date_cols].agg(["min", "max", lambda s: s.isna().sum()]).T.rename(
    columns={"<lambda>": "n_missing"}
)

,min,max,n_missing
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0


In [4]:
for name in ["items", "payments", "reviews"]:
    counts = d[name].groupby("order_id").size()
    print(name,
          "| orders covered:", counts.shape[0],
          "| max rows per order:", counts.max(),
          "| orders with >1 row:", (counts > 1).sum())

print("orders.order_id unique:", orders["order_id"].is_unique)

items | orders covered: 98666 | max rows per order: 21 | orders with >1 row: 9803
payments | orders covered: 99440 | max rows per order: 29 | orders with >1 row: 2961
reviews | orders covered: 98673 | max rows per order: 3 | orders with >1 row: 547
orders.order_id unique: True
